# Universidad Peruana de Ciencias Aplicadas
## Curso: Procesamiento de Imágenes
## Práctica Calificada 02 — Semestre 2026-01
**Docente:** PhD. Carlos Fernando Montoya Cubas

---

## Segmentación avanzada mediante espacios de color y umbralización multiespacio

**Objetivo:** Aislar con alta precisión todas las frutas (plátanos, manzana, naranja/toronja y pera) de su fondo con patrones azulados, verdosos, morados y beige, mediante análisis técnico multiespacio.

---

## Integrantes
- Cielo Luwidka Chavez Merino -U20191E443
- Fabrizio Bussalleu Salcedo - U202315655
- Robert Leonardo Alvarado Valle - U202111912


## Instalación e importación de librerías

Se utilizan OpenCV para procesamiento de imágenes, NumPy para operaciones matriciales, Matplotlib para visualización y scikit-image para morfología avanzada.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import ndimage
from skimage import morphology, measure, filters
from skimage.morphology import disk, remove_small_objects, binary_closing, binary_opening
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.titleweight'] = 'bold'

print('✔ Librerías importadas correctamente')
print(f'  OpenCV:       {cv2.__version__}')
print(f'  NumPy:        {np.__version__}')

: 

---
# PARTE 1: Carga y exploración de la imagen

En esta sección cargamos la imagen desde Google Drive o Colab, verificamos su estructura y la visualizamos correctamente. OpenCV lee imágenes en orden **BGR**, por lo que se debe convertir a **RGB** antes de mostrarlas con Matplotlib.

In [ ]:

RUTA = 'frutas.jpeg'

img_bgr = cv2.imread(RUTA)

if img_bgr is None:
    raise FileNotFoundError(f'No se pudo leer la imagen en: {RUTA}')

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

alto, ancho, canales = img_rgb.shape
print('=== INFORMACIÓN DE LA IMAGEN ===')
print(f'  Dimensiones : {ancho} × {alto} píxeles')
print(f'  Canales     : {canales}  (RGB)')
print(f'  Tipo de dato: {img_rgb.dtype}')
print(f'  Rango valor : [{img_rgb.min()}, {img_rgb.max()}]')
print(f'  Tamaño total: {img_rgb.nbytes / 1024:.1f} KB')

plt.figure(figsize=(7, 5))
plt.imshow(img_rgb)
plt.title('Imagen Original — Frutas variadas sobre fondo con patrones', fontsize=11, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print('\n✔ Imagen cargada y visualizada correctamente')

**Observaciones iniciales:** La imagen muestra plátanos (amarillo intenso), una manzana (roja), una naranja/toronja (naranja) y una pera (amarillo-naranja moteado) sobre un fondo con patrones geométricos de colores fríos (azul, verde, morado) y zonas beige. El desafío está en que algunos tonos del fondo (amarillo-beige) pueden confundirse visualmente con regiones de las frutas. La conversión BGR→RGB es crítica para interpretar correctamente los canales en el análisis posterior.

---
# PARTE 2: Descomposición en espacios de color

Analizaremos la imagen en tres espacios de color complementarios: **RGB**, **CMY** y **HSI**. Cada espacio ofrece una perspectiva diferente sobre la separabilidad entre frutas y fondo.

In [ ]:
R = img_rgb[:, :, 0].astype(np.float64) / 255.0
G = img_rgb[:, :, 1].astype(np.float64) / 255.0
B = img_rgb[:, :, 2].astype(np.float64) / 255.0

# Fórmula: C = 1-R,  M = 1-G,  Y = 1-B
C = 1.0 - R    # Cyan
M = 1.0 - G    # Magenta
Y = 1.0 - B    # Yellow

eps = 1e-8

I = (R + G + B) / 3.0

# Saturacion
minRGB = np.minimum(np.minimum(R, G), B)
S = 1.0 - (3.0 / (R + G + B + eps)) * minRGB
S = np.clip(S, 0, 1)

# Matiz H
numerador = 0.5 * ((R - G) + (R - B))
denominador = np.sqrt((R - G)**2 + (R - B) * (G - B)) + eps
theta = np.arccos(np.clip(numerador / denominador, -1, 1))

H = np.where(B <= G, theta, 2 * np.pi - theta)
H = H / (2 * np.pi)   # Normalizar

print('✔ Canales RGB, CMY y HSI calculados')
print(f'  R: min={R.min():.3f}, max={R.max():.3f}, media={R.mean():.3f}')
print(f'  G: min={G.min():.3f}, max={G.max():.3f}, media={G.mean():.3f}')
print(f'  B: min={B.min():.3f}, max={B.max():.3f}, media={B.mean():.3f}')
print(f'  S: min={S.min():.3f}, max={S.max():.3f}, media={S.mean():.3f}')
print(f'  I: min={I.min():.3f}, max={I.max():.3f}, media={I.mean():.3f}')

**Nota técnica:** La intensidad HSI es el promedio aritmético de R, G, B. La saturación se define como la distancia al eje acromático (1 − min/media_ponderada). El matiz H se calcula mediante la proyección trigonométrica de los vectores de color, produciendo ángulos en [0, 2π] que luego se normalizan.

In [ ]:
canales_info = [
    (R, 'R — Rojo',       'Reds_r'),
    (G, 'G — Verde',      'Greens_r'),
    (B, 'B — Azul',       'Blues_r'),
    (C, 'C — Cyan (1-R)', 'Blues'),
    (M, 'M — Magenta (1-G)', 'Purples'),
    (Y, 'Y — Yellow (1-B)', 'YlGn'),
    (H, 'H — Matiz HSI',  'hsv'),
    (S, 'S — Saturación', 'plasma'),
    (I, 'I — Intensidad', 'gray'),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
fig.suptitle('Descomposición en espacios de color: RGB · CMY · HSI', fontsize=13, fontweight='bold', y=1.01)

for ax, (datos, titulo, cmap) in zip(axes.ravel(), canales_info):
    im = ax.imshow(datos, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle('Análisis por canal: imagen + histograma', fontsize=12, fontweight='bold')

grupos = [
    [(R, 'R', 'Reds_r'), (G, 'G', 'Greens_r'), (B, 'B', 'Blues_r')],
    [(C, 'C', 'Blues'),   (M, 'M', 'Purples'),   (Y, 'Y', 'YlGn')],
    [(H, 'H', 'hsv'),    (S, 'S', 'plasma'),     (I, 'I', 'gray')],
]

colores_hist = ['red', 'green', 'blue', 'cyan', 'magenta', 'goldenrod', 'purple', 'orange', 'gray']
idx_color = 0

for fila, grupo in enumerate(grupos):
    for col_base, (datos, nombre, cmap) in enumerate(grupo):
        ax_img  = axes[fila][col_base * 2]
        ax_hist = axes[fila][col_base * 2 + 1]

        # Imagen del canal
        ax_img.imshow(datos, cmap=cmap, vmin=0, vmax=1)
        ax_img.set_title(nombre, fontsize=9, fontweight='bold')
        ax_img.axis('off')

        # Histograma
        ax_hist.hist(datos.ravel(), bins=80, color=colores_hist[idx_color], alpha=0.75, edgecolor='none')
        ax_hist.set_xlim(0, 1)
        ax_hist.set_xlabel('Intensidad', fontsize=7)
        ax_hist.set_ylabel('Frecuencia', fontsize=7)
        ax_hist.tick_params(labelsize=6)
        ax_hist.set_title(f'Histograma {nombre}', fontsize=8)
        idx_color += 1

plt.tight_layout()
plt.show()

### Análisis técnico comparativo de canales

#### Espacio RGB

| Canal | ¿Resalta frutas? | ¿Confunde con fondo? | Sombras | Ruido | Utilidad |
|-------|-----------------|---------------------|---------|-------|----------|
| **R (Rojo)** | Alto — frutas cálidas tienen R elevado | Parcial — zonas beige del fondo también tienen R alto | Moderado — sombras reducen R | Bajo | Alta: buen discriminador inicial |
| **G (Verde)** | Bajo — frutas no son verdes | Alto — fondo teal/verdoso se confunde | Moderado | Bajo | Baja: inverso útil como penalizador |
| **B (Azul)** | Muy bajo — frutas tienen B mínimo | Excelente — fondo azulado/morado tiene B alto | Bajo | Bajo | Muy alta como **penalizador de fondo** |

#### Espacio CMY

| Canal | ¿Resalta frutas? | ¿Confunde con fondo? | Sombras | Ruido | Utilidad |
|-------|-----------------|---------------------|---------|-------|----------|
| **C (Cyan=1-R)** | Bajo — complementario de R | Alto en fondo azul | Moderado | Bajo | Penalizador útil |
| **M (Magenta=1-G)** | Parcial — frutas tienen M>0.5 | Confunde con patrones no verdes | Moderado | Bajo | Complementario |
| **Y (Yellow=1-B)** | Alto — frutas tienen Y alto (poco azul) | Parcial — zona beige también | Moderado | Bajo | Alta: colores cálidos → Y elevado |

#### Espacio HSI

| Canal | ¿Resalta frutas? | ¿Confunde con fondo? | Sombras | Ruido | Utilidad |
|-------|-----------------|---------------------|---------|-------|----------|
| **H (Matiz)** | Parcial — amarillos/naranja agrupados | Inestable en zonas oscuras/desaturadas | Alto — sombras distorsionan H | Alto | Limitada sola, útil en combinación |
| **S (Saturación)** | Alto — frutas son muy saturadas | Parcial — patrones del fondo también saturados | Bajo | Moderado | Alta: las frutas tienen cromás definido |
| **I (Intensidad)** | Muy bajo — no discrimina color | Fondo y frutas similares en brillo | Alto — sombras oscurecen I | Bajo | Baja como discriminador, útil como corrector |

**Conclusión del análisis:** Ningún canal aislado es suficiente. Los mejores candidatos individuales son **R** (realza frutas cálidas) y **B** (penaliza el fondo azul). En CMY, **Y** resalta frutas bien. La **saturación S** distingue objetos coloridos pero no discrimina fondo de fruta por sí sola. Esto motiva una combinación algebraica personalizada.

---
# PARTE 3: Estrategia simple inicial

Seleccionamos el canal **R** (rojo) como mejor discriminador individual, por su alta respuesta en todos los tipos de fruta (amarillo, naranja, rojo). Aplicamos umbralización de Otsu para determinar el umbral óptimo de forma estadística.

In [ ]:

# Convertir canal R para usar Otsu
R_uint8 = (R * 255).astype(np.uint8)

umbral_otsu, mascara_simple_raw = cv2.threshold(
    R_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

umbral_norm = umbral_otsu / 255.0
mascara_simple = (mascara_simple_raw > 0).astype(np.uint8)

print(f'Umbral de Otsu en canal R: {umbral_otsu:.0f}/255  ({umbral_norm:.3f} normalizado)')
print(f'Píxeles clasificados como fruta: {mascara_simple.sum():,}')
print(f'Porcentaje de imagen: {100*mascara_simple.sum()/(alto*ancho):.1f}%')

resultado_simple = img_rgb.copy()
resultado_simple[mascara_simple == 0] = 0   # fondo → negro

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('PARTE 3 — Estrategia Simple: Canal R + Otsu', fontsize=12, fontweight='bold')

axes[0].imshow(img_rgb)
axes[0].set_title('Imagen original', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(R, cmap='Reds_r', vmin=0, vmax=1)
axes[1].axhline(y=alto//2, color='cyan', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].set_title(f'Canal R (Otsu = {umbral_norm:.2f})', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mascara_simple, cmap='gray')
axes[2].set_title('Máscara simple (Otsu en R)', fontweight='bold')
axes[2].axis('off')

axes[3].imshow(resultado_simple)
axes[3].set_title('Resultado simple', fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()

# Histograma del canal R con umbral
plt.figure(figsize=(8, 3))
plt.hist(R.ravel(), bins=100, color='red', alpha=0.7, label='Canal R')
plt.axvline(x=umbral_norm, color='black', linewidth=2, linestyle='--',
            label=f'Umbral Otsu = {umbral_norm:.3f}')
plt.fill_betweenx([0, plt.ylim()[1] if plt.ylim()[1] > 0 else 50000],
                  umbral_norm, 1.0, alpha=0.15, color='green', label='Zona fruta')
plt.xlabel('Valor normalizado R')
plt.ylabel('Frecuencia')
plt.title('Histograma canal R con umbral de Otsu')
plt.legend()
plt.tight_layout()
plt.show()

### Análisis de la estrategia simple

**Fortalezas:** El canal R detecta la mayor parte de frutas de colores cálidos. El umbral de Otsu es estadísticamente óptimo para distribuciones bimodales.

**Limitaciones críticas:**
- **Falsos positivos:** Zonas beige-amarillas del fondo tienen R similar a las frutas.
- **Falsos negativos:** Sombras reducen R por debajo del umbral, fragmentando la máscara.
- **Sin información semántica:** No distingue qué tipo de fruta es cada región.

Estas limitaciones motivan una estrategia de **segmentación individual por fruta** en el espacio HSV, que permite aislar cada fruta de forma independiente antes de combinarlas en una máscara global de alta precisión.

---
# PARTE 4: Segmentación individual por fruta en espacio HSV

## Fundamento cromático del enfoque

El espacio HSV (Hue–Saturation–Value) de OpenCV separa ortogonalmente el **tono cromático** (H: 0–180°), la **pureza del color** (S: 0–255) y el **brillo** (V: 0–255). Esta separación facilita la definición de rangos compactos por fruta:

| Fruta | Rango H (OpenCV) | Condición S | Condición V |
|-------|-----------------|-------------|-------------|
| **Manzana** | [0–10°] ∪ [158–180°] | ≥ 100 | ≥ 40 |
| **Naranja/Toronja** | [10–22°] | ≥ 130 | ≥ 60 |
| **Plátano** | [20–33°] | ≥ 120 | ≥ 80 |
| **Pera** | [30–68°] | 25–130 | ≥ 80 |

La **supresión de fondo** se realiza excluyendo píxeles con canal B (RGB) > 0.40 (fondos azulados/morados) o saturación HSV < 50 (fondos beige/neutros), antes de aplicar cualquier rango individual.

In [ ]:
# Convertir a espacio HSV de OpenCV (H:0-180, S:0-255, V:0-255)
img_hsv_cv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
H_cv = img_hsv_cv[:,:,0].astype(np.float64)  # 0-180
S_cv = img_hsv_cv[:,:,1].astype(np.float64)  # 0-255
V_cv = img_hsv_cv[:,:,2].astype(np.float64)  # 0-255

area_min = int(0.002 * alto * ancho)

# Supresión de fondo: excluir azul/morado (B alto) y beige/neutro (S_HSV bajo)
mask_no_fondo = (B < 0.40) & (S_cv > 50)

def rango_hsv(h_lo, h_hi, s_lo, s_hi, v_lo, v_hi):
    lo = np.array([h_lo, s_lo, v_lo], dtype=np.uint8)
    hi = np.array([h_hi, s_hi, v_hi], dtype=np.uint8)
    return cv2.inRange(img_hsv_cv, lo, hi) > 0

# Manzana (roja): H ≈ 0° o 180° — el rojo envuelve en el círculo cromático
mask_manzana_raw = rango_hsv(0,  10, 100, 255, 40, 255) | rango_hsv(158, 180, 100, 255, 40, 255)

# Naranja/Toronja: H en zona naranja [10-22°]
mask_naranja_raw = rango_hsv(10, 22, 130, 255, 60, 255)

# Plátano: H amarillo [20-33°], alta saturación para separar del beige
mask_platano_raw = rango_hsv(20, 33, 120, 255, 80, 255)

# Pera: H amarillo-verde [30-68°], saturación moderada (distingue de plátano y fondo)
mask_pera_raw = rango_hsv(30, 68, 25, 130, 80, 255)

# Aplicar supresión de fondo
mask_manzana_filt = mask_manzana_raw & mask_no_fondo
mask_naranja_filt = mask_naranja_raw & mask_no_fondo
mask_platano_filt = mask_platano_raw & mask_no_fondo
mask_pera_filt    = mask_pera_raw    & mask_no_fondo

frutas_info = [
    ('Manzana (roja)',        mask_manzana_raw, mask_manzana_filt, 'Reds',    '#c0392b'),
    ('Naranja/Toronja',      mask_naranja_raw, mask_naranja_filt, 'Oranges', '#d35400'),
    ('Plátano (amarillo)',    mask_platano_raw, mask_platano_filt, 'YlOrBr',  '#d4ac0d'),
    ('Pera (verde-amarilla)', mask_pera_raw,    mask_pera_filt,   'YlGn',    '#27ae60'),
]

print('=== MÁSCARAS INICIALES POR FRUTA ===')
print(f'{"Fruta":<28} {"Raw":>10}   {"Filtrada":>10}  {"% imagen":>8}')
print('─' * 64)
for nombre, m_raw, m_filt, _, _ in frutas_info:
    n_raw  = int(m_raw.sum())
    n_filt = int(m_filt.sum())
    pct    = 100 * n_filt / (alto * ancho)
    print(f'{nombre:<28} {n_raw:>10,}   {n_filt:>10,}  {pct:>7.2f}%')
print(f'\nÁrea mínima de componente: {area_min:,} px (0.2% del área total)')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('PARTE 4 — Rangos HSV individuales por fruta', fontsize=13, fontweight='bold')

for col, (nombre, m_raw, m_filt, cmap, color_hist) in enumerate(frutas_info):
    # Fila 0: imagen original con zona seleccionada por rango HSV raw resaltada
    overlay = img_rgb.copy().astype(np.float32)
    overlay[~m_raw] = overlay[~m_raw] * 0.28
    axes[0][col].imshow(overlay.astype(np.uint8))
    axes[0][col].set_title(f'{nombre}\n(rango HSV raw)', fontsize=8, fontweight='bold')
    axes[0][col].axis('off')

    # Fila 1: máscara binaria tras supresión de fondo
    axes[1][col].imshow(m_filt.astype(np.uint8), cmap=cmap)
    n = int(m_filt.sum())
    axes[1][col].set_title(f'Máscara filtrada\n{n:,} px', fontsize=8, fontweight='bold')
    axes[1][col].axis('off')

    # Fila 2: distribución del matiz H dentro de la máscara filtrada
    ax_h = axes[2][col]
    if m_filt.sum() > 0:
        h_vals = H_cv[m_filt]
        ax_h.hist(h_vals, bins=60, color=color_hist, alpha=0.82, edgecolor='none')
        ax_h.axvline(h_vals.mean(), color='black', linewidth=1.5, linestyle='--',
                     label=f'Media = {h_vals.mean():.1f}°')
        ax_h.legend(fontsize=6)
    ax_h.set_xlim(0, 180)
    ax_h.set_xlabel('Matiz H (OpenCV, 0–180°)', fontsize=7)
    ax_h.set_ylabel('Frecuencia', fontsize=7)
    ax_h.tick_params(labelsize=6)
    ax_h.set_title('Distribución H en máscara filtrada', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def refinar_mascara_fruta(mask_bool, radio_ap=3, radio_ci=10, min_sz=None):
    """Pipeline morfológico adaptado para una fruta individual."""
    if min_sz is None:
        min_sz = area_min
    m = binary_opening(mask_bool, disk(radio_ap))
    m = remove_small_objects(m, min_size=min_sz)
    m = binary_closing(m, disk(radio_ci))
    m = ndimage.binary_fill_holes(m)
    m = remove_small_objects(m, min_size=min_sz)
    return m

mask_manzana_ref = refinar_mascara_fruta(mask_manzana_filt, radio_ap=4, radio_ci=12)
mask_naranja_ref = refinar_mascara_fruta(mask_naranja_filt, radio_ap=4, radio_ci=12)
mask_platano_ref = refinar_mascara_fruta(mask_platano_filt, radio_ap=3, radio_ci=11)
mask_pera_ref    = refinar_mascara_fruta(mask_pera_filt,    radio_ap=2, radio_ci=10,
                                          min_sz=area_min // 2)

nombres_frutas = ['Manzana', 'Naranja', 'Plátano', 'Pera']
mascaras_pre   = [mask_manzana_filt, mask_naranja_filt, mask_platano_filt, mask_pera_filt]
mascaras_ref   = [mask_manzana_ref,  mask_naranja_ref,  mask_platano_ref,  mask_pera_ref]
cmaps_frutas   = ['Reds', 'Oranges', 'YlOrBr', 'YlGn']

print('=== REFINAMIENTO MORFOLÓGICO POR FRUTA ===')
print(f'{"Fruta":<10} {"Pre-morf.":>12}  {"Post-morf.":>12}  {"Cambio":>8}')
print('─' * 50)
for nombre, m_pre, m_ref in zip(nombres_frutas, mascaras_pre, mascaras_ref):
    n_pre = int(m_pre.sum())
    n_ref = int(m_ref.sum())
    print(f'{nombre:<10} {n_pre:>12,}  {n_ref:>12,}  {n_ref - n_pre:>+8,}')

# Visualización: antes vs. después por fruta
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Refinamiento morfológico individual por fruta', fontsize=12, fontweight='bold')

for i in range(4):
    axes[0][i].imshow(mascaras_pre[i].astype(np.uint8), cmap=cmaps_frutas[i])
    axes[0][i].set_title(f'{nombres_frutas[i]}\npre-morfología · {int(mascaras_pre[i].sum()):,} px',
                          fontsize=8, fontweight='bold')
    axes[0][i].axis('off')

    axes[1][i].imshow(mascaras_ref[i].astype(np.uint8), cmap=cmaps_frutas[i])
    axes[1][i].set_title(f'{nombres_frutas[i]}\npost-morfología · {int(mascaras_ref[i].sum()):,} px',
                          fontsize=8, fontweight='bold')
    axes[1][i].axis('off')

plt.tight_layout()
plt.show()

### Justificación técnica del refinamiento morfológico individual

Cada fruta tiene morfología y textura distintas, por lo que se aplican parámetros diferenciados:

- **Manzana y Naranja (apertura r=4, cierre r=12):** Frutas de superficie lisa y forma compacta. La apertura r=4 elimina ruido sin erosionar bordes curvos. El cierre r=12 reconecta regiones separadas por sombras internas.
- **Plátano (apertura r=3, cierre r=11):** Forma elongada. Una apertura menor preserva la elongación sin segmentar en subregiones. El cierre es ligeramente menor para no deformar la curvatura.
- **Pera (apertura r=2, cierre r=10, min_sz reducido):** Textura moteada amarillo-verde. La apertura mínima evita destruir el patrón de moteado. El tamaño mínimo es la mitad del general para no eliminar manchas legítimas de la pera.

---
# PARTE 5: Combinación de máscaras individuales

Las cuatro máscaras refinadas se combinan mediante **unión lógica (OR)**, produciendo una máscara global que incluye todos los píxeles clasificados como cualquier fruta. Un refinamiento morfológico final corrige huecos inter-fruta y residuos aislados.

In [ ]:
# Unión lógica (OR) de las cuatro máscaras individuales refinadas
mascara_combinada = (mask_manzana_ref | mask_naranja_ref |
                     mask_platano_ref | mask_pera_ref)

# Refinamiento morfológico final sobre la máscara global combinada
mascara_final_bool = binary_closing(mascara_combinada, disk(8))
mascara_final_bool = ndimage.binary_fill_holes(mascara_final_bool)
mascara_final_bool = remove_small_objects(mascara_final_bool, min_size=area_min)
mascara_final      = mascara_final_bool.astype(np.uint8)

print('=== COMBINACIÓN PROGRESIVA DE MÁSCARAS (OR acumulativo) ===')
acum = np.zeros((alto, ancho), dtype=bool)
for nombre, mask in zip(nombres_frutas, mascaras_ref):
    acum = acum | mask
    n = int(acum.sum())
    print(f'  + {nombre:<10}: {n:>10,} px  ({100*n/(alto*ancho):.2f}%)')
print(f'  Refinamiento final : {int(mascara_final.sum()):>10,} px  ({100*mascara_final.sum()/(alto*ancho):.1f}%)')

In [ ]:
# Imagen coloreada por fruta: cada fruta recibe su color representativo
colores_frutas_rgb = [
    np.array([192,  57,  43]),  # manzana  — rojo
    np.array([211,  84,   0]),  # naranja  — naranja
    np.array([212, 172,  13]),  # plátano  — amarillo
    np.array([ 39, 174,  96]),  # pera     — verde
]

img_coloreada = np.zeros_like(img_rgb)
for mask, color in zip(mascaras_ref, colores_frutas_rgb):
    img_coloreada[mask] = color

resultado_final = img_rgb.copy()
resultado_final[mascara_final == 0] = 0

# Figura principal: 4 frutas individuales + mapa cromático + máscara combinada + final
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('PARTE 5 — Segmentación individual + combinación final',
             fontsize=13, fontweight='bold')

titulos_viz = ['Manzana', 'Naranja/Toronja', 'Plátano', 'Pera']
for i in range(4):
    overlay = img_rgb.copy().astype(np.float32)
    overlay[~mascaras_ref[i]] = overlay[~mascaras_ref[i]] * 0.18
    axes[0][i].imshow(overlay.astype(np.uint8))
    n = int(mascaras_ref[i].sum())
    axes[0][i].set_title(f'{titulos_viz[i]}\n({n:,} px · {100*n/(alto*ancho):.1f}%)',
                          fontsize=9, fontweight='bold')
    axes[0][i].axis('off')

axes[1][0].imshow(img_coloreada)
axes[1][0].set_title('Mapa cromático por fruta\n(color = identidad de fruta)', fontsize=9, fontweight='bold')
axes[1][0].axis('off')

axes[1][1].imshow(mascara_combinada.astype(np.uint8), cmap='gray')
axes[1][1].set_title(f'Máscara combinada (OR)\n{int(mascara_combinada.sum()):,} px',
                      fontsize=9, fontweight='bold')
axes[1][1].axis('off')

axes[1][2].imshow(mascara_final, cmap='gray')
axes[1][2].set_title(f'Máscara final refinada\n{int(mascara_final.sum()):,} px',
                      fontsize=9, fontweight='bold')
axes[1][2].axis('off')

axes[1][3].imshow(resultado_final)
axes[1][3].set_title('Resultado final\n(frutas sobre fondo negro)', fontsize=9, fontweight='bold')
axes[1][3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Proceso acumulativo: construcción de la máscara global fruta por fruta
acum = np.zeros((alto, ancho), dtype=bool)
pasos_acum = []
for nombre, mask in zip(nombres_frutas, mascaras_ref):
    acum = acum | mask
    pasos_acum.append((acum.copy().astype(np.uint8), f'+ {nombre}'))
pasos_acum.append((mascara_final, 'Refinamiento final'))

cmaps_acum = ['Reds', 'Oranges', 'YlOrBr', 'YlGn', 'gray']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Proceso de combinación acumulativa — OR progresivo por fruta',
             fontsize=11, fontweight='bold')

for ax, (mask, titulo), cmap in zip(axes, pasos_acum, cmaps_acum):
    ax.imshow(mask, cmap=cmap)
    n = int(mask.sum())
    ax.set_title(f'{titulo}\n{n:,} px ({100*n/(alto*ancho):.1f}%)',
                 fontsize=8, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

**Propiedades de la combinación por unión:**

La unión OR es la operación correcta porque las frutas son objetos físicamente distintos y no superpuestos. Los píxeles en zonas de borde entre frutas adyacentes quedan incluidos en ambas máscaras individuales sin generar inconsistencias, ya que la unión los incorpora una sola vez.

**Refinamiento final:** El cierre global r=8 une posibles huecos entre frutas que se tocan físicamente; el relleno de huecos completa el interior; la eliminación de componentes pequeñas descarta residuos del fondo que sobrevivieron a todos los filtros previos.

---
# PARTE 6: Comparación final — Simple vs. Segmentación por fruta

In [ ]:
# Refinar máscara simple para comparación justa
mascara_simple_bool = mascara_simple.astype(bool)
mascara_simple_ref  = binary_closing(mascara_simple_bool, disk(8))
mascara_simple_ref  = ndimage.binary_fill_holes(mascara_simple_ref)
mascara_simple_ref  = remove_small_objects(mascara_simple_ref, min_size=area_min)
mascara_simple_ref  = mascara_simple_ref.astype(np.uint8)

resultado_simple_ref = img_rgb.copy()
resultado_simple_ref[mascara_simple_ref == 0] = 0

# Diferencias entre estrategias
diff_fp = (mascara_simple_ref.astype(int) - mascara_final.astype(int)).clip(0)  # simple incluye, por-fruta no
diff_fn = (mascara_final.astype(int) - mascara_simple_ref.astype(int)).clip(0)  # por-fruta incluye, simple no

fig = plt.figure(figsize=(18, 10))
fig.suptitle('PARTE 6 — Comparación: Estrategia Simple vs. Segmentación por Fruta',
             fontsize=13, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 5, figure=fig, wspace=0.08, hspace=0.30)

paneles = [
    (img_rgb,              'Original',                       None,   0, 0),
    (mascara_simple_ref,   'Máscara Simple\n(Otsu en R)',    'gray', 0, 1),
    (resultado_simple_ref, 'Resultado Simple',               None,   0, 2),
    (mascara_final,        'Máscara por Fruta\n(HSV + OR)',  'gray', 0, 3),
    (resultado_final,      'Resultado por Fruta',            None,   0, 4),
]

for datos, titulo, cmap, fila, col in paneles:
    ax = fig.add_subplot(gs[fila, col])
    ax.imshow(datos, cmap=cmap) if cmap else ax.imshow(datos)
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.axis('off')

ax_fp = fig.add_subplot(gs[1, 0:2])
ax_fp.imshow(diff_fp, cmap='Reds')
ax_fp.set_title('Simple incluye / por-fruta NO\n(falsos positivos eliminados)',
                fontsize=9, fontweight='bold')
ax_fp.axis('off')

ax_mid = fig.add_subplot(gs[1, 2])
ax_mid.imshow(img_coloreada)
ax_mid.set_title('Mapa por fruta\n(referencia)', fontsize=9, fontweight='bold')
ax_mid.axis('off')

ax_fn = fig.add_subplot(gs[1, 3:5])
ax_fn.imshow(diff_fn, cmap='Greens')
ax_fn.set_title('Por-fruta incluye / simple NO\n(zonas recuperadas)',
                fontsize=9, fontweight='bold')
ax_fn.axis('off')

plt.savefig('comparacion_final.png', dpi=120, bbox_inches='tight')
plt.show()
print('✔ Figura comparativa guardada como comparacion_final.png')

In [ ]:
total_px = alto * ancho

px_simple    = int(mascara_simple_ref.sum())
px_por_fruta = int(mascara_final.sum())
interseccion = int(np.logical_and(mascara_simple_ref, mascara_final).sum())
union        = int(np.logical_or(mascara_simple_ref, mascara_final).sum())
iou          = interseccion / (union + 1e-8)
solo_simple    = int(diff_fp.sum())
solo_por_fruta = int(diff_fn.sum())

print('════════════════════════════════════════════════════════')
print('   ESTADÍSTICAS COMPARATIVAS DE SEGMENTACIÓN')
print('════════════════════════════════════════════════════════')
print(f'  Imagen total                  : {total_px:>10,} px')
print(f'  Máscara simple (Otsu R)       : {px_simple:>10,} px  ({100*px_simple/total_px:.2f}%)')
print(f'  Máscara por fruta (HSV+OR)    : {px_por_fruta:>10,} px  ({100*px_por_fruta/total_px:.2f}%)')
print(f'  Intersección                  : {interseccion:>10,} px  ({100*interseccion/total_px:.2f}%)')
print(f'  Unión                         : {union:>10,} px  ({100*union/total_px:.2f}%)')
print(f'  IoU (Jaccard)                 : {iou:.4f}')
print(f'  Solo simple (FP estimado)     : {solo_simple:>10,} px  ({100*solo_simple/total_px:.2f}%)')
print(f'  Solo por-fruta (mejoras)      : {solo_por_fruta:>10,} px  ({100*solo_por_fruta/total_px:.2f}%)')
print('════════════════════════════════════════════════════════')
print()
print('=== DESGLOSE POR FRUTA (estrategia individual) ===')
print(f'  {"Fruta":<25} {"Píxeles":>10}  {"% imagen":>8}')
print('  ' + '─' * 48)
for nombre, mask in zip(nombres_frutas, mascaras_ref):
    n = int(mask.sum())
    print(f'  {nombre:<25} {n:>10,}  {100*n/total_px:>7.2f}%')

### Comparación de estrategias

| Criterio | Estrategia Simple (Otsu en R) | Segmentación por Fruta (HSV + OR) |
|----------|-------------------------------|-----------------------------------|
| **Identificación individual** | No — solo "fruta" vs "fondo" | Sí — cada fruta segmentada aparte |
| **Falsos positivos** | Altos — fondo beige confundido | Bajos — rangos HSV selectivos por clase |
| **Robustez a sombras** | Baja — R cae en sombras | Moderada — V permite tolerancia al brillo |
| **Complejidad** | Baja — un canal, un umbral | Moderada — 4 rangos + morfología individual |
| **Continuidad de máscara** | Fragmentada en sombras | Continua tras morfología individual |
| **Información semántica** | Ninguna | Qué fruta es cada región |

**¿Por qué la segmentación por fruta es más robusta?** Porque cada rango HSV está diseñado para el perfil cromático específico de cada fruta, minimizando la contaminación cruzada con el fondo. El matiz H es parcialmente invariante a variaciones de brillo (sombras), lo que el canal R en RGB no garantiza. La unión OR captura la cobertura total sin perder la selectividad individual.

---
# PARTE 7: Respuestas a las preguntas obligatorias

## Pregunta 1: ¿Por qué ciertos canales resultaron más útiles que otros?

El matiz **H** del espacio HSV resultó el canal más útil para segmentación por fruta porque codifica el tipo de color de forma parcialmente invariante al brillo: un plátano en sombra y uno iluminado tienen el mismo H pero diferente V. El canal R en RGB no tiene esta propiedad.

La saturación **S** fue el segundo canal más útil: separa colores vividos (frutas) de colores apagados (fondo beige) y distingue el plátano (S alto) de la pera (S moderado) dentro del mismo rango de H.

El canal **B** (azul, RGB) siguió siendo útil como penalizador de fondo en la máscara de supresión previa: los fondos azulados/morados tienen B alto, y excluirlos antes de la umbralización HSV reduce drásticamente los falsos positivos.

Los canales **G** e **I** no se utilizaron directamente porque no poseen variación sistemática entre frutas y fondo de colores fríos.

## Pregunta 2: ¿Por qué una sola representación fue insuficiente?

Un solo canal no puede separar las cuatro frutas entre sí ni del fondo simultáneamente. La pera y el plátano comparten rango de H; la distinción requiere la dimensión S. La manzana roja y el fondo rojo-beige comparten rango de R; la distinción requiere H y S conjuntamente.

La segmentación por fruta opera en el espacio 2D (H, S) por cada clase, con V como filtro mínimo de brillo. Esto ofrece una capacidad discriminativa fundamentalmente mayor que cualquier proyección unidimensional.

## Pregunta 3: ¿Qué fundamento técnico respalda la segmentación por fruta?

La técnica de umbralización en espacio HSV por rangos es equivalente a una **clasificación por región en el espacio de color**: se define un paralelepípedo (caja) en el espacio (H, S, V) que delimita el volumen cromático de cada fruta.

Este enfoque tiene fundamento en la **teoría de clasificadores de mínima distancia en espacios perceptuales**: HSV separa la información cromática (H, S) de la fotométrica (V), lo que hace que las regiones de decisión por fruta sean más compactas y estables que en RGB. La unión OR final es equivalente a la unión de los conjuntos de decisión individuales, sin necesidad de asumir distribuciones probabilísticas.

## Pregunta 4: ¿Qué limitaciones persisten?

1. **Solapamiento en H:** Plátano (H=[20–33°]) y Naranja (H=[10–22°]) solapan en H=[20–22°]; píxeles en esa zona pueden asignarse a ambas máscaras.
2. **Pera vs. fondo verde-amarillo:** El fondo tiene zonas verde-amarillas que caen en el rango de la pera; la supresión por S_HSV<50 las reduce pero no las elimina del todo.
3. **Sombras intensas:** V muy bajo hace que H sea inestable incluso en HSV.
4. **Reflejos especulares:** Brillos blancos en manzana/naranja tienen S≈0 y quedan excluidos; el relleno morfológico los corrige parcialmente.
5. **Calibración manual:** Los rangos HSV se determinaron visualmente y pueden no generalizarse a otras iluminaciones o frutas de diferente madurez.

## Pregunta 5: ¿Cómo podría mejorarse aún más la segmentación?

1. **Calibración automática de rangos:** Usar k-means en el espacio (H, S) sobre regiones etiquetadas manualmente para determinar rangos óptimos por fruta.
2. **Clasificador bayesiano:** Modelar la distribución de color de cada fruta como una gaussiana 2D en (H, S) y usar el clasificador MAP en lugar de cajas de umbral duro.
3. **Análisis de textura:** La pera tiene textura moteada característica; descriptores LBP locales mejorarían su detección.
4. **Watershed con semillas por fruta:** Usar los centroides de las máscaras individuales como semillas de watershed para refinar bordes de contacto entre frutas adyacentes.
5. **Espacio CIELAB:** El canal a* (verde–rojo) y b* (azul–amarillo) ofrecerían rangos más compactos y perceptualmente uniformes por fruta.

In [ ]:
# Resumen visual completo
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('RESUMEN VISUAL COMPLETO — Práctica Calificada 02\n'
             'Segmentación por fruta individual en HSV + combinación OR',
             fontsize=12, fontweight='bold')

axes[0][0].imshow(img_rgb)
axes[0][0].set_title('① Imagen original', fontweight='bold')
axes[0][0].axis('off')

axes[0][1].imshow(img_coloreada)
axes[0][1].set_title('② Segmentación individual\n(color = identidad de fruta)', fontweight='bold')
axes[0][1].axis('off')

axes[0][2].imshow(mascara_simple_ref, cmap='gray')
axes[0][2].set_title('③ Máscara simple\n(Otsu en canal R)', fontweight='bold')
axes[0][2].axis('off')

axes[0][3].imshow(resultado_simple_ref)
axes[0][3].set_title('④ Resultado simple', fontweight='bold')
axes[0][3].axis('off')

axes[1][0].imshow(mascara_final, cmap='gray')
axes[1][0].set_title('⑤ Máscara por fruta\n(HSV individual + OR + morfología)', fontweight='bold')
axes[1][0].axis('off')

axes[1][1].imshow(resultado_final)
axes[1][1].set_title('⑥ Resultado por fruta — FINAL', fontweight='bold')
axes[1][1].axis('off')

axes[1][2].imshow(diff_fp, cmap='Reds')
axes[1][2].set_title('⑦ Falsos positivos eliminados\n(simple incluye, por-fruta no)', fontweight='bold')
axes[1][2].axis('off')

axes[1][3].imshow(diff_fn, cmap='Greens')
axes[1][3].set_title('⑧ Zonas recuperadas\n(por-fruta incluye, simple no)', fontweight='bold')
axes[1][3].axis('off')

plt.tight_layout()
plt.savefig('resumen_final.png', dpi=120, bbox_inches='tight')
plt.show()
print('✔ Figura de resumen guardada como resumen_final.png')

## Conclusiones técnicas

La presente práctica demostró que la segmentación semántica de múltiples objetos requiere un enfoque estructurado que combine conocimiento cromático específico por clase con operaciones morfológicas adaptadas a la forma de cada objeto.

**Primera conclusión:** La estrategia de segmentación individual por fruta en el espacio HSV produce máscaras más selectivas que la umbralización global en un canal único. Al operar con rangos bidimensionales (H, S) por clase, se reduce drásticamente la contaminación del fondo beige y los falsos positivos típicos del canal R.

**Segunda conclusión:** La separación ortogonal de cromatismo y brillo en HSV es fundamentalmente superior a RGB para este tipo de tarea. El matiz H es invariante a sombras suaves (solo V cae), lo que permite mantener la clasificación correcta en zonas de iluminación variable, resolviendo directamente la principal limitación del canal R.

**Tercera conclusión:** El diseño de morfología diferenciada por fruta —radios de apertura y cierre adaptados a la forma esperada de cada objeto— es más preciso que un pipeline morfológico uniforme. La pera moteada requiere menos apertura que la manzana lisa; el plátano elongado requiere menos cierre que la naranja compacta.

**Cuarta conclusión:** La combinación por unión lógica (OR) es la operación natural para integrar máscaras de objetos no superpuestos. La máscara resultante preserva la cobertura total y, mediante el mapa cromático, mantiene la información semántica (qué fruta es cada región), que la estrategia simple no puede ofrecer.

**Reflexión final:** Este ejercicio ilustra la transición desde la segmentación puramente fotométrica (canal único + Otsu) hacia la segmentación semántica basada en conocimiento del dominio (rangos HSV por clase + morfología adaptativa). La clave del éxito es el análisis previo del espacio de color de cada objeto de interés, que permite diseñar clasificadores simples pero específicos, y combinarlos de forma coherente para obtener una máscara global de alta calidad.

---
*Práctica Calificada 02 — Procesamiento de Imágenes — UPC 2026-01*